# SCRFD-34G gegen SCRFD-10G auf einem Ausschnitt

**Frage:** Erkennt das größere Modell **SCRFD-34G** Gesichter an der Stelle von VID_76, an der **SCRFD-10G** nichts (oder zu wenig) findet?

**Vorgehen:** Es ist dieselbe Pipeline wie im 10G-Notebook, nur mit einer anderen Modelldatei:
1. Zwei Modelle laden (10G, 34G).
2. Nur den **Ausschnitt** mit dem Problem durch 34G laufen lassen (die 10G-Ergebnisse liegen schon in `FACES_10G_FILE`).
3. Die rohen Erkennungen vergleichen (Score je Frame, getrennt für linke und rechte Ansicht).
4. Dieselbe Nachbearbeitung (Tracking, Lücken, Glätten) auf beide anwenden und zählen, in wie vielen Frames **keine Box** ist.
5. Optional: beide Ausschnitte als Video rendern und selbst ansehen.

**Datenschutz:** Das Notebook gibt nur Zahlen aus. Die Videos landen in `data/processed` (gitignored). Vor einem Commit alle Ausgaben löschen.

| | SCRFD-10G (bisher) | SCRFD-34G (neu) |
|---|---|---|
| Datei | `det_10g.onnx` aus `buffalo_l` | `34g_dynamic.onnx` (umgebaut, siehe unten) |
| Größe der Datei | 17 MB | 39 MB |
| Ausgänge | 9 (mit 5 Gesichtspunkten) | 6 (nur Boxen, Punkte brauchen wir nicht) |
| WIDER FACE „Hard“ (laut InsightFace-README, VGA) | 82,80 (10G_KPS) | 85,29 |
| Zeit pro Frame bei 1280 × 736 (dieser Laptop, CPU) | ca. 0,9 s | ca. 3,3 s |

„Hard“ sind schwierige Gesichter (klein, gedreht, verdeckt). Der Abstand ist also nur etwa 2,5 Punkte. Ob er im Anfall etwas bringt, soll dieser Test zeigen.

## 0. Woher die 34G-Datei kommt (einmalig, schon erledigt)

- **Quelle:** `34g.onnx` aus dem Hugging-Face-Repo `RuteNL/SCRFD-face-detection-ONNX`. Das ist eine **Umwandlung der offiziellen Gewichte** von InsightFace in ONNX durch Dritte. Das offizielle Repo liefert nur `.pth`-Gewichte.
- **Prüfsumme (SHA-256) der heruntergeladenen Datei:** `5fb6dce6dbdb3a2866ec401297fca008fbd48ea768ed2edb33546a8d7c3ae39d`
- **Problem:** Diese Datei akzeptiert nur Bilder von genau **640 × 640** (die Größe ist im Netz fest eingebaut). Wir arbeiten mit 1280 × 736.
- **Lösung:** `scripts/make_scrfd34g_dynamic.py` ersetzt die festen Zahlen durch „nimm die tatsächliche Größe“. Das Skript prüft, dass bei 640 × 640 dieselben Werte herauskommen wie im Original (Abweichung ca. 2·10⁻⁶).
  Aufruf: `uv run --isolated --no-project --with onnx --with onnxruntime --with numpy python scripts/make_scrfd34g_dynamic.py models/scrfd_34g/34g.onnx models/scrfd_34g/34g_dynamic.onnx`
- **Lizenz:** Das Repo nennt Apache-2.0. Die Gewichte stammen aber von InsightFace und sind laut deren Hinweisen nur für nicht-kommerzielle Forschung gedacht. Für den klinischen Einsatz klären. Der Ordner `models/` ist deshalb nicht im Repo.

## 1. Setup und Einstellungen

In [1]:
import pickle
import sys
import time
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort

PROJECT_ROOT = Path.cwd().parent
MODEL_10G = PROJECT_ROOT / "models" / "buffalo_l" / "buffalo_l" / "det_10g.onnx"
MODEL_34G = PROJECT_ROOT / "models" / "scrfd_34g" / "34g_dynamic.onnx"
INPUT_VIDEO = PROJECT_ROOT / "data" / "raw" / "PAT_1" / "VID_76.AVI"
FACES_10G_FILE = PROJECT_ROOT / "data" / "interim" / "PAT_1" / "VID_76_faces_1280.pkl"   # fertige 10G-Erkennung (ganzes Video)

# Der Ausschnitt: Frames CLIP_START bis CLIP_END-1. Bei 25 FPS sind 150 Frames = 6 Sekunden.
# 8100-8250 enthält den Übergang in die längste Lücke von 10G (ab Frame 8123 rechts, ab 8133 ganz ohne Box).
# Mit 34G dauert ein Frame ca. 3,3 s: 150 Frames = ca. 9 Minuten.
CLIP_START, CLIP_END = 8100, 8250

FACES_34G_FILE = PROJECT_ROOT / "data" / "interim" / "PAT_1" / f"VID_76_faces_34g_{CLIP_START}_{CLIP_END}.pkl"
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "PAT_1"

FPS = 25
SPLIT = 960      # Grenze zwischen linker (RGB) und rechter (schwarz-weiß) Ansicht in Pixeln

for name, p in [("Modell 10G", MODEL_10G), ("Modell 34G", MODEL_34G), ("Video", INPUT_VIDEO), ("10G-Erkennungen", FACES_10G_FILE)]:
    print(f"{name}: {'gefunden' if p.exists() else 'FEHLT'}")

Modell 10G: gefunden
Modell 34G: gefunden
Video: gefunden
10G-Erkennungen: gefunden


## 2. Beide Modelle ansehen

Wichtig sind Eingang und Ausgänge:
- **10G:** Höhe und Breite sind offen (`?`), 9 Ausgänge (je 3 für Scores, Boxen, Gesichtspunkte).
- **34G:** nach dem Umbau ebenfalls offen, aber nur **6 Ausgänge** (Scores und Boxen für die drei Maßstäbe 8, 16, 32).

Unsere `SCRFD`-Klasse benutzt nur die ersten 6 Ausgänge, deshalb funktioniert sie für beide.

In [2]:
for name, path in [("10G", MODEL_10G), ("34G", MODEL_34G)]:
    s = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    print(f"--- {name}: {path.name}, {path.stat().st_size / 1e6:.1f} MB")
    for i in s.get_inputs():
        print("  Eingang:", i.name, i.shape)
    for o in s.get_outputs():
        print("  Ausgang:", o.name, o.shape)

--- 10G: det_10g.onnx, 16.9 MB
  Eingang: input.1 [1, 3, '?', '?']
  Ausgang: 448 [12800, 1]
  Ausgang: 471 [3200, 1]
  Ausgang: 494 [800, 1]
  Ausgang: 451 [12800, 4]
  Ausgang: 474 [3200, 4]
  Ausgang: 497 [800, 4]
  Ausgang: 454 [12800, 10]
  Ausgang: 477 [3200, 10]
  Ausgang: 500 [800, 10]
--- 34G: 34g_dynamic.onnx, 39.2 MB
  Eingang: input.1 [1, 3, 'H', 'W']
  Ausgang: score_8 [1, 'score_8_d1', 1]
  Ausgang: score_16 [1, 'score_16_d1', 1]
  Ausgang: score_32 [1, 'score_32_d1', 1]
  Ausgang: bbox_8 [1, 'bbox_8_d1', 4]
  Ausgang: bbox_16 [1, 'bbox_16_d1', 4]
  Ausgang: bbox_32 [1, 'bbox_32_d1', 4]


## 3. Der Detektor (unverändert aus dem 10G-Notebook)

Ablauf pro Bild: auf die Netzgröße verkleinern (Letterbox mit schwarzem Rand), normalisieren, Netz laufen lassen, aus den Gitterpunkten Boxen berechnen, überlappende Boxen mit NMS zusammenfassen und auf die Originalgröße zurückrechnen.

In [3]:
class SCRFD:
    def __init__(self, model_path, input_size=(1280, 736), score_threshold=0.5, nms_threshold=0.4, pad_value=127):
        self.pad_value = pad_value
        self.in_w, self.in_h = input_size
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.strides = [8, 16, 32]
        self.num_anchors = 2

        self.session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        print("Ausgaben des Netzes:", len(self.output_names), "(6 = nur Boxen, 9 = mit Gesichtspunkten)")

    def detect(self, img_bgr):
        ih, iw = img_bgr.shape[:2]

        # 1. Letterbox
        scale = min(self.in_w / iw, self.in_h / ih)
        new_w, new_h = int(iw * scale), int(ih * scale)
        canvas = np.full((self.in_h, self.in_w, 3), self.pad_value, dtype=np.uint8)
        canvas[:new_h, :new_w] = cv2.resize(img_bgr, (new_w, new_h))

        # 2. Netz: (Pixel - 127.5) / 128, BGR -> RGB
        blob = cv2.dnn.blobFromImage(canvas, 1.0 / 128, (self.in_w, self.in_h),
                                     (127.5, 127.5, 127.5), swapRB=True)
        outs = self.session.run(self.output_names, {self.input_name: blob})

        boxes, scores = [], []
        for idx, stride in enumerate(self.strides):
            s = outs[idx].reshape(-1)                      # Scores
            d = outs[idx + 3].reshape(-1, 4) * stride      # Abstände in Pixeln

            # 3. Gitterpunkte dieses Maßstabs, je 2 Anker
            gh, gw = self.in_h // stride, self.in_w // stride
            ys, xs = np.mgrid[:gh, :gw]
            centers = np.stack([xs, ys], axis=-1).astype(np.float32).reshape(-1, 2) * stride
            centers = np.repeat(centers, self.num_anchors, axis=0)

            for k in np.where(s >= self.score_threshold)[0]:
                cx, cy = centers[k]
                x1, y1 = cx - d[k, 0], cy - d[k, 1]
                x2, y2 = cx + d[k, 2], cy + d[k, 3]
                boxes.append([x1 / scale, y1 / scale, (x2 - x1) / scale, (y2 - y1) / scale])
                scores.append(float(s[k]))

        if not boxes:
            return 0, np.empty((0, 5), dtype=np.float32)

        # 4. NMS
        idxs = np.array(cv2.dnn.NMSBoxes(boxes, scores, self.score_threshold, self.nms_threshold)).reshape(-1)
        faces = np.array([[*boxes[i], scores[i]] for i in idxs], dtype=np.float32)
        return len(faces), faces

## 4. Hilfsfunktionen: Tracking, Lücken, Glätten (unverändert aus dem 10G-Notebook)

Genau dieselben Funktionen und Einstellungen wie bei 10G. Nur so ist der Vergleich fair. Hier nichts ändern.

In [4]:
def iou(a, b):
    """Überlappung zweier Boxen (x, y, w, h): 0 = keine, 1 = identisch."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def inter_over_smaller(a, b):
    """Anteil der kleineren Box, der in der anderen liegt (0 bis 1)."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    return inter / max(1e-9, min(aw * ah, bw * bh))


def near_track(box, tracks, pad=0.5, min_frac=0.3):
    """True, wenn mindestens min_frac der Box im verpixelten Bereich einer Spur liegt."""
    x, y, w, h = box
    for t in tracks:
        tx, ty, tw, th = t["box"]
        rx0, ry0 = tx - pad * tw, ty - pad * th
        rx1, ry1 = tx + tw + pad * tw, ty + th + pad * th
        ix = max(0, min(x + w, rx1) - max(x, rx0))
        iy = max(0, min(y + h, ry1) - max(y, ry0))
        if ix * iy >= min_frac * w * h:
            return True
    return False


def track_with_ids(all_faces, start_threshold=0.5, low_threshold=0.2, match_iou=0.3,
                   max_age=8, damping=0.85, near_threshold=None):
    tracks, result, next_id = [], [], 0
    for faces in all_faces:
        cands = [(tuple(float(v) for v in f[:4]), float(f[4])) for f in faces if f[4] >= low_threshold]

        for t in tracks:                                    # erwartete Position
            x, y, w, h = t["box"]
            t["pred"] = (x + t["v"][0], y + t["v"][1], w, h)

        pairs = sorted(((iou(t["pred"], box), ti, ci)
                        for ti, t in enumerate(tracks) for ci, (box, s) in enumerate(cands)),
                       reverse=True)
        used_t, used_c = set(), set()
        for ov, ti, ci in pairs:
            if ov < match_iou:
                break
            if ti in used_t or ci in used_c:
                continue
            t, new = tracks[ti], cands[ci][0]
            steps = t["age"] + 1
            mvx = (new[0] - t["det"][0]) / steps
            mvy = (new[1] - t["det"][1]) / steps
            t["v"] = (0.5 * t["v"][0] + 0.5 * mvx, 0.5 * t["v"][1] + 0.5 * mvy)
            t["box"], t["det"], t["age"] = new, new, 0
            used_t.add(ti)
            used_c.add(ci)

        for ti, t in enumerate(tracks):                     # nicht gefunden: weiterlaufen
            if ti not in used_t:
                t["box"] = t["pred"]
                t["v"] = (t["v"][0] * damping, t["v"][1] * damping)
                t["age"] += 1
        tracks = [t for t in tracks if t["age"] <= max_age]

        for ci, (box, s) in enumerate(cands):               # neue Spuren
            if ci in used_c:
                continue
            if s < start_threshold:
                # NEU: schwacher Fund nur, wenn er an einer bestehenden Spur liegt
                if near_threshold is None or s < near_threshold or not near_track(box, tracks):
                    continue
            if any(inter_over_smaller(box, t["box"]) >= 0.5 for t in tracks):
                continue
            tracks.append({"id": next_id, "box": box, "det": box, "v": (0.0, 0.0), "age": 0})
            next_id += 1

        result.append([(t["id"], t["box"]) for t in tracks])
    return result


def bridge_gaps(tracked, gap_max=40, dist_factor=1.5):
    n = len(tracked)
    per_id = {}
    for f, items in enumerate(tracked):
        for tid, box in items:
            per_id.setdefault(tid, {})[f] = box
    first = {tid: min(d) for tid, d in per_id.items()}
    last = {tid: max(d) for tid, d in per_id.items()}

    def center(b):
        return (b[0] + b[2] / 2, b[1] + b[3] / 2)

    added = [[] for _ in range(n)]
    for b_id in sorted(per_id, key=lambda t: first[t]):
        fb = first[b_id]
        boxb = per_id[b_id][fb]
        best, best_dist = None, None
        for a_id in per_id:                                  # früher beendete Spur suchen
            if a_id == b_id:
                continue
            gap = fb - last[a_id] - 1
            if gap < 1 or gap > gap_max:
                continue
            boxa = per_id[a_id][last[a_id]]
            (ax, ay), (bx, by) = center(boxa), center(boxb)
            dist = ((ax - bx) ** 2 + (ay - by) ** 2) ** 0.5
            if dist <= dist_factor * max(boxa[2], boxb[2]) and (best is None or dist < best_dist):
                best, best_dist = a_id, dist
        if best is not None:                                 # Lücke linear füllen
            la = last[best]
            boxa = per_id[best][la]
            for f in range(la + 1, fb):
                t = (f - la) / (fb - la)
                box = tuple(boxa[i] + t * (boxb[i] - boxa[i]) for i in range(4))
                added[f].append((best, box))
    return [tracked[f] + added[f] for f in range(n)]


def smooth_tracks(tracked, k=3, k_back=5):
    n = len(tracked)
    per_id = {}                                              # Kennung -> {Frame: Box}
    for f, items in enumerate(tracked):
        for tid, box in items:
            per_id.setdefault(tid, {})[f] = box

    out = [[] for _ in range(n)]
    for tid, d in per_id.items():
        first = min(d)
        for f in range(max(0, first - k_back), first):       # Anfang nach vorne verlängern
            d[f] = d[first]
        frames = sorted(d)
        arr = np.array([d[f] for f in frames], dtype=float)  # Spalten: x, y, w, h
        cx, cy = arr[:, 0] + arr[:, 2] / 2, arr[:, 1] + arr[:, 3] / 2
        for j, f in enumerate(frames):
            lo, hi = max(0, j - k), min(len(frames), j + k + 1)
            w, h = arr[lo:hi, 2].max(), arr[lo:hi, 3].max()  # größte Größe im Fenster
            mx, my = cx[lo:hi].mean(), cy[lo:hi].mean()      # gemittelte Mitte
            out[f].append((mx - w / 2, my - h / 2, w, h))
    return out


def dedupe(boxes_per_frame, overlap=0.6):
    out = []
    for boxes in boxes_per_frame:
        kept = []
        for b in sorted(boxes, key=lambda b: b[2] * b[3], reverse=True):
            if all(inter_over_smaller(b, kk) < overlap for kk in kept):
                kept.append(b)
        out.append(kept)
    return out

## 5. Hilfen zum Lesen des Ausschnitts

`cap.set(POS_FRAMES, …)` kann bei MPEG-2 ungenau sein. Deshalb liest `skip_to` alle Frames davor nur ein und wirft sie weg. Das dauert etwa eine Minute, aber die Frame-Nummer stimmt dann sicher.

`detect_clip` erkennt nur die Frames des Ausschnitts und speichert alle 25 Frames. Nach einem Abbruch (Laptop zugeklappt) setzt es dort fort.

In [5]:
def skip_to(cap, index):
    """Springt zu Frame `index`, indem die Frames davor gelesen und verworfen werden."""
    for _ in range(index):
        if not cap.grab():
            raise RuntimeError("Video ist kürzer als erwartet")


def detect_clip(input_path, detector, start, end, out_file, save_every=25):
    """Erkennt die Frames start..end-1. Ergebnis: Liste mit einem Array [x, y, w, h, score] pro Frame."""
    faces = []
    if Path(out_file).exists():
        with open(out_file, "rb") as f:
            faces = pickle.load(f)
        print("Fortgesetzt ab Ausschnitt-Frame", len(faces))

    def save():
        Path(out_file).parent.mkdir(parents=True, exist_ok=True)
        with open(out_file, "wb") as f:
            pickle.dump(faces, f)

    cap = cv2.VideoCapture(str(input_path))
    t0 = time.time()
    try:
        skip_to(cap, start + len(faces))
        for _ in range(start + len(faces), end):
            ok, frame = cap.read()
            if not ok:
                break
            _, found = detector.detect(frame)
            faces.append(found)
            if len(faces) % save_every == 0:
                save()
                print(f"{len(faces)}/{end - start} Frames, {time.time() - t0:.0f} s")
    finally:
        save()
        cap.release()
    return faces

## 6. Kontrolle: stimmt die Frame-Nummer?

Wir erkennen den ersten Frame des Ausschnitts noch einmal mit 10G und vergleichen mit der gespeicherten 10G-Erkennung für diese Nummer. Sind sie gleich, liegen 10G und 34G auf denselben Bildern und der Vergleich ist sauber.

In [6]:
with open(FACES_10G_FILE, "rb") as f:
    faces_10g_all = pickle.load(f)
print("10G-Erkennungen für", len(faces_10g_all), "Frames geladen")

detector_10g = SCRFD(MODEL_10G, input_size=(1280, 736), score_threshold=0.2, pad_value=0)

cap = cv2.VideoCapture(str(INPUT_VIDEO))
skip_to(cap, CLIP_START)
ok, frame = cap.read()
cap.release()

_, neu = detector_10g.detect(frame)
alt = faces_10g_all[CLIP_START]
del frame                      # Bild nicht im Speicher/Notebook behalten

if len(alt) == 0:
    gleich = len(neu) == 0
else:
    gleich = len(neu) == len(alt) and np.allclose(neu[np.argsort(-neu[:, 4])], alt[np.argsort(-alt[:, 4])], atol=1e-2)
print(f"Frame {CLIP_START}: neu {len(neu)} Funde, gespeichert {len(alt)} Funde -> gleich: {gleich}")
assert gleich, "Frame-Nummer passt nicht zur gespeicherten Erkennung - Ausschnitt nicht vergleichbar"

10G-Erkennungen für 8759 Frames geladen
Ausgaben des Netzes: 9 (6 = nur Boxen, 9 = mit Gesichtspunkten)
Frame 8100: neu 0 Funde, gespeichert 0 Funde -> gleich: True


## 7. 34G auf dem Ausschnitt laufen lassen

Gleiche Einstellungen wie bei 10G: Eingabe 1280 × 736, schwarzer Rand, Schwelle 0,2 (alles ab 0,2 wird gespeichert, die Auswahl kommt später).
Dauer: ca. 3,3 s pro Frame, bei 150 Frames etwa 9 Minuten plus eine Minute zum Vorspulen.

In [7]:
detector_34g = SCRFD(MODEL_34G, input_size=(1280, 736), score_threshold=0.2, pad_value=0)

faces_34g = detect_clip(INPUT_VIDEO, detector_34g, CLIP_START, CLIP_END, FACES_34G_FILE)
faces_10g = faces_10g_all[CLIP_START:CLIP_END]
print("Frames im Ausschnitt: 34G", len(faces_34g), "| 10G", len(faces_10g))
assert len(faces_34g) == len(faces_10g)

Ausgaben des Netzes: 6 (6 = nur Boxen, 9 = mit Gesichtspunkten)
25/150 Frames, 115 s
50/150 Frames, 186 s
75/150 Frames, 286 s
100/150 Frames, 354 s
125/150 Frames, 434 s
150/150 Frames, 512 s
Frames im Ausschnitt: 34G 150 | 10G 150


## 8. Vergleich der rohen Erkennungen

Pro Frame und Ansicht zählt der **beste Score** (0 = nichts gefunden). Das Tracking später braucht mindestens 0,5, um eine Spur zu starten, und 0,2, um sie weiterzuführen.

Lies die Tabelle so:
- **≥ 0,5:** sicher genug für den Start einer Spur.
- **≥ 0,3 / ≥ 0,2:** schwacher Fund, nützt nur einer schon laufenden Spur.
- Wenn 34G bei ≥ 0,5 deutlich mehr Frames hat, erkennt es dort wirklich besser. Wenn nur bei ≥ 0,2 mehr Frames hinzukommen, ist das vor allem mehr Rauschen.

In [8]:
def in_half(box, half):
    cx = box[0] + box[2] / 2
    return cx < SPLIT if half == "links" else cx >= SPLIT


def best_per_frame(faces, half):
    return np.array([max((float(b[4]) for b in fr if in_half(b, half)), default=0.0) for fr in faces])


n = CLIP_END - CLIP_START
print(f"Ausschnitt: Frame {CLIP_START}-{CLIP_END - 1} (Sekunde {CLIP_START / FPS:.1f}-{CLIP_END / FPS:.1f}), {n} Frames\n")
for half in ("links", "rechts"):
    for name, fa in (("10G", faces_10g), ("34G", faces_34g)):
        best = best_per_frame(fa, half)
        print(f"{half:6s} {name}: Frames mit Fund >=0.5: {(best >= 0.5).sum():3d} | >=0.3: {(best >= 0.3).sum():3d} | "
              f">=0.2: {(best >= 0.2).sum():3d} | Median des Bestwerts: {np.median(best):.2f}")
    print()

print("Verlauf (bester Score je Ansicht, Maximum über je 10 Frames):")
print("Frame  | links 10G  34G | rechts 10G  34G")
for s in range(0, n, 10):
    row = []
    for half in ("links", "rechts"):
        for fa in (faces_10g, faces_34g):
            row.append(best_per_frame(fa[s:s + 10], half).max())
    print(f"{CLIP_START + s:6d} |       {row[0]:.2f} {row[1]:.2f} |       {row[2]:.2f}  {row[3]:.2f}")

Ausschnitt: Frame 8100-8249 (Sekunde 324.0-330.0), 150 Frames

links  10G: Frames mit Fund >=0.5:   2 | >=0.3:  14 | >=0.2:  56 | Median des Bestwerts: 0.00
links  34G: Frames mit Fund >=0.5:   2 | >=0.3: 108 | >=0.2: 146 | Median des Bestwerts: 0.34

rechts 10G: Frames mit Fund >=0.5:   5 | >=0.3:  14 | >=0.2:  61 | Median des Bestwerts: 0.00
rechts 34G: Frames mit Fund >=0.5:   4 | >=0.3:  17 | >=0.2: 109 | Median des Bestwerts: 0.22

Verlauf (bester Score je Ansicht, Maximum über je 10 Frames):
Frame  | links 10G  34G | rechts 10G  34G
  8100 |       0.24 0.40 |       0.29  0.47
  8110 |       0.36 0.42 |       0.58  0.63
  8120 |       0.57 0.65 |       0.58  0.68
  8130 |       0.22 0.37 |       0.21  0.31
  8140 |       0.00 0.38 |       0.21  0.27
  8150 |       0.20 0.40 |       0.00  0.30
  8160 |       0.24 0.38 |       0.00  0.42
  8170 |       0.40 0.47 |       0.26  0.43
  8180 |       0.28 0.48 |       0.32  0.30
  8190 |       0.43 0.44 |       0.28  0.25
  8200 |       

## 9. Nachbearbeitung und Zählung der Frames ohne Box

Dieselbe Kette (Tracking → Lücken füllen → Glätten → Doppelte entfernen) auf beide Erkennungen. Zusätzlich je eine Variante mit **niedrigerer Startschwelle 0,3** statt 0,5. Das ist die einfache Alternative ohne neues Modell. So sehen wir, ob 34G wirklich mehr bringt oder ob 10G mit einer anderen Einstellung schon reicht.

- **Frames ohne Box:** dort ist nichts verpixelt (das Ziel ist 0).
- **Fläche:** wie viel vom Bild verpixelt wird (mit Rand 0,5 um jede Box). Niedrigere Schwellen erzeugen oft mehr Boxen und damit mehr Fläche.

In [9]:
def pipeline(faces, start_threshold=0.5):
    t = track_with_ids(faces, start_threshold=start_threshold, low_threshold=0.2, max_age=8, near_threshold=0.2)
    t = bridge_gaps(t, gap_max=40)
    return dedupe(smooth_tracks(t, k=3, k_back=5))


def area_percent(boxes_per_frame, padding=0.5, shape=(1080, 1920)):
    h, w = shape
    out = []
    for boxes in boxes_per_frame:
        mask = np.zeros((h, w), dtype=bool)
        for x, y, bw, bh in boxes:
            mask[int(max(0, y - padding * bh)):int(min(h, y + bh + padding * bh)),
                 int(max(0, x - padding * bw)):int(min(w, x + bw + padding * bw))] = True
        out.append(100.0 * mask.mean())
    return np.array(out)


variants = {
    "10G, Start 0,5 (bisheriger Stand)": (faces_10g, 0.5),
    "10G, Start 0,3": (faces_10g, 0.3),
    "34G, Start 0,5": (faces_34g, 0.5),
    "34G, Start 0,3": (faces_34g, 0.3),
}
results = {}
print(f"{'Variante':36s} | ohne Box: gesamt  links  rechts | Fläche Median / Max")
for name, (fa, thr) in variants.items():
    boxes = pipeline(fa, start_threshold=thr)
    results[name] = boxes
    none_all = sum(1 for b in boxes if not b)
    none_l = sum(1 for b in boxes if not any(in_half(x, "links") for x in b))
    none_r = sum(1 for b in boxes if not any(in_half(x, "rechts") for x in b))
    a = area_percent(boxes)
    print(f"{name:36s} |           {none_all:5d} {none_l:6d} {none_r:7d} | {np.median(a):.1f} % / {a.max():.1f} %")

Variante                             | ohne Box: gesamt  links  rechts | Fläche Median / Max
10G, Start 0,5 (bisheriger Stand)    |             129    140     139 | 0.0 % / 12.7 %
10G, Start 0,3                       |              49     49      66 | 2.8 % / 13.4 %
34G, Start 0,5                       |             131    140     141 | 0.0 % / 13.0 %
34G, Start 0,3                       |               0      0      65 | 15.6 % / 62.2 %


## 10. Optional: beide Ausschnitte als Video

Ausgabe in `data/processed/PAT_1/`. Gerendert werden die Varianten mit Startschwelle 0,5, also „10G bisher“ und „34G“. Die Videos enthalten das Original mit verpixelten Gesichtern. Sieh dir besonders die Frames an, in denen die Tabelle oben Lücken zeigt.

In [10]:
def render_clip(input_path, output_path, boxes_per_frame, start, block_px=22, padding=0.5):
    """Wie render_mask aus dem 10G-Notebook, aber nur für den Ausschnitt ab Frame `start`."""
    cap = cv2.VideoCapture(str(input_path))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    bw, bh = max(1, width // block_px), max(1, height // block_px)
    written = 0
    try:
        skip_to(cap, start)
        for boxes in boxes_per_frame:
            ok, frame = cap.read()
            if not ok:
                break
            mask = np.zeros((height, width), dtype=bool)
            for x, y, w, h in boxes:
                mask[int(max(0, y - padding * h)):int(min(height, y + h + padding * h)),
                     int(max(0, x - padding * w)):int(min(width, x + w + padding * w))] = True
            if mask.any():
                small = cv2.resize(frame, (bw, bh), interpolation=cv2.INTER_AREA)
                pix = cv2.resize(small, (width, height), interpolation=cv2.INTER_NEAREST)
                frame[mask] = pix[mask]
            writer.write(frame)
            written += 1
    finally:
        cap.release()
        writer.release()
    return written


RENDER_CLIPS = True
if RENDER_CLIPS:
    for name, key in [("10G", "10G, Start 0,5 (bisheriger Stand)"), ("34G", "34G, Start 0,5")]:
        out = OUT_DIR / f"VID_76_clip_{CLIP_START}_{CLIP_END}_{name}.mp4"
        print(name, render_clip(INPUT_VIDEO, out, results[key], CLIP_START), "Frames ->", out.name)

10G 150 Frames -> VID_76_clip_8100_8250_10G.mp4
34G 150 Frames -> VID_76_clip_8100_8250_34G.mp4


## 11. Auswertung (nach dem Lauf ausfüllen)

**Wie du die Zahlen einordnest:**
- **34G findet bei ≥ 0,5 deutlich mehr Frames** und die Frames ohne Box sinken: Das größere Modell hilft hier. Dann lohnt sich ein Lauf über das ganze Video (ca. 3,8-fache Zeit, also grob 8 Stunden statt 2,2) oder nur über die Problemstellen.
- **Nur die Variante „Start 0,3“ hilft (bei 10G und 34G ähnlich):** Das Modell ist nicht das Problem, sondern die Startschwelle. Das lässt sich ohne neue Erkennung testen.
- **Auch 34G findet dort nichts (Score < 0,2):** Das Gesicht ist für diese Modellfamilie unerkennbar (starke Drehung, Verdeckung). Dann braucht es einen anderen Ansatz (Kopf- oder Pose-Erkennung) oder eine Rückfallregel.
- Nur ein **Ausschnitt** von 6 Sekunden: Das ist ein Hinweis, kein Beweis. Prüfe danach ein bis zwei weitere Lücken (Frames 7789–7902, 6742–6801, 5576–5621).

**Ergebnis:** _(hier eintragen)_